In [1]:
import sys, os, json, time
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar

from tcr_pipeline.data_io import load_anonymized_rds, load_rdata
from tcr_pipeline.normalization import tmm_normalize
from tcr_pipeline import nb_glm as nb

In [2]:
HEALTHY_RDS  = "../data/Healthy-Data-Anonymized.rds"     # <- adjust
PATIENTS_RDS = "../data/BRIOME-Batch-1-Data-Anonymized.rds"    # <- adjust
TEST_PIDS    = ["BR054", "BR037"]                         # patients to test
EXCLUDE_SAMPLES = []          # e.g. [("HDBR06_SEQTR_ReSeq", 21)] after QC in cell 3
N_JOBS       = 8              # parallel workers for the trajectory test
MIN_DETECT_DISP = 2           # detections per clone for dispersion fitting
MIN_DETECT_TEST = 3           # detections per clone for the slope test
ALPHA        = 0.05
N_V2         = 3000           # clones per design in the missing-data comparison
N_V3         = 4000           # clones per patient in the end-to-end test
OUT_DIR      = "../results/nb_v2"

In [3]:
healthy_raw = load_anonymized_rds(HEALTHY_RDS)
patients_raw = load_anonymized_rds(PATIENTS_RDS)

In [4]:
patients_raw = patients_raw[
    patients_raw['patient'].isin(TEST_PIDS)]

In [5]:
def sample_table(d):
    g = d.groupby(['patient', 'day'])
    t = g.agg(D_t=('D_t', 'first'), lib_post=('lib_post', 'first'),
              c_t=('c_t', 'first'), n_clones=('clono', 'size'),
              top_clone_freq=('freq_in', 'max'))
    t['removed_frac'] = 1 - t['lib_post'] / t['D_t']
    t['n_at_cutoff'] = g.apply(lambda s: int((s['count'] == s['c_t']).sum()))
    t['top_clone_freq'] = t['top_clone_freq'] / 100
    return t

samples_h = sample_table(healthy_raw)
samples_p = sample_table(patients_raw)
display(samples_h.round(4))
display(samples_p.round(4))

D_t  lib_post   c_t  n_clones  top_clone_freq  \
patient            day                                                         
HD106_SEQTR        0    20776057.0  19064838  21.0    191146          0.0109   
                   46   20271816.0  18814334  21.0    166686          0.0118   
                   82   13155700.0  11923023  14.0    196560          0.0098   
                   131  16998893.0  14866483  17.0    213759          0.0088   
HD197_SEQTR        0    16017072.0  13598350  17.0    213417          0.0064   
                   46   15062825.0  12942781  16.0    189060          0.0111   
                   82   17667150.0  16076784  18.0    179473          0.0093   
                   131  17706840.0  15864008  18.0    184846          0.0077   
HDBR01_SEQTR_ReSeq 0    10155538.0   9797482  11.0    101763          0.0032   
                   21   12854243.0  12286013  13.0    133280          0.0030   
                   42   21302289.0  20439637  22.0    109420          0.0026   
                   126  13816805.0  13264378  14.0    115549          0.0029   
HDBR02_SEQTR_ReSeq 0     5380856.0   5207497   6.0     87988          0.1940   
                   48    8857000.0   8690147   9.0     53280          0.1563   
                   71    6669562.0   6384434   7.0     93623          0.2588   
HDBR03_SEQTR_ReSeq 0    10650237.0  10393489  11.0     71638          0.0377   
                   21    4888783.0   4643232   5.0    157045          0.0410   
                   35    6397513.0   6206839   7.0    100372          0.0293   
                   66    5034447.0   4832580   6.0    118041          0.0209   
                   84    5390104.0   5176791   6.0     98678          0.0167   
HDBR06_SEQTR_ReSeq 0    10704506.0  10547250  11.0     43115          0.0458   
                   21    5073105.0   5069548   6.0       706          0.2425   
                   82   11483748.0  11209119  12.0     65536          0.0411   
                   105  10809851.0  10440607  11.0     85011          0.0359   

                        removed_frac  n_at_cutoff  
patient            day                             
HD106_SEQTR        0          0.0824         4479  
                   46         0.0719         3717  
                   82         0.0937         8096  
                   131        0.1254         8492  
HD197_SEQTR        0          0.1510         9626  
                   46         0.1407         9165  
                   82         0.0900         5585  
                   131        0.1041         5972  
HDBR01_SEQTR_ReSeq 0          0.0353         2760  
                   21         0.0442         3304  
                   42         0.0405         1489  
                   126        0.0400         2344  
HDBR02_SEQTR_ReSeq 0          0.0322         6222  
                   48         0.0188         1910  
                   71         0.0428         7165  
HDBR03_SEQTR_ReSeq 0          0.0241         1904  
                   21         0.0502        15346  
                   35         0.0298         4373  
                   66         0.0401         7887  
                   84         0.0396         5714  
HDBR06_SEQTR_ReSeq 0          0.0147          979  
                   21         0.0007           83  
                   82         0.0239         1516  
                   105        0.0342         2362

D_t  lib_post   c_t  n_clones  top_clone_freq  \
patient day                                                         
BR037   0     5198802.0   4883559   6.0     88873          0.0048   
        28    3278684.0   3123716   4.0    123647          0.0488   
        119   4587433.0   4391807   5.0     67530          0.0037   
        189   6536904.0   6244275   7.0     44655          0.0041   
        218   5843098.0   5491121   6.0     78871          0.0037   
        364   8673209.0   8422731   9.0     72570          0.0038   
BR054   0     5352048.0   5151362   6.0     99188          0.0272   
        19   11294031.0  10356050  12.0    154386          0.0250   
        68    5067147.0   4738367   6.0    105314          0.0202   
        141   2926693.0   2901235   3.0     30093          0.1914   
        187   4155331.0   3975392   5.0     84805          0.0188   
        327   4411067.0   4296450   5.0     30551          0.0259   

             removed_frac  n_at_cutoff  
patient day                             
BR037   0          0.0606         5795  
        28         0.0473        12489  
        119        0.0426         6341  
        189        0.0448         4068  
        218        0.0602         6609  
        364        0.0289         1965  
BR054   0          0.0375         5080  
        19         0.0831         6022  
        68         0.0649         7874  
        141        0.0087         5543  
        187        0.0433         6198  
        327        0.0260         4385